# 📊 Complete COVID-19 Forecasting with DeepAR

## 🎯 Project Goal

**Predict COVID-19 cases in the United States using real data and DeepAR**

This is a complete, end-to-end pipeline:
1. Load real COVID-19 data (cases, deaths, mobility, vaccines)
2. Preprocess and merge multiple data sources
3. Engineer features for better predictions
4. Train a DeepAR model with exogenous features
5. Generate 14-day forecasts with uncertainty
6. Comprehensive evaluation
7. Scenario analysis (bonus)

**Data Sources**:
- 📈 Johns Hopkins University COVID-19 data (cases, deaths)
- 🚗 Google Mobility data (movement patterns)
- 💉 CDC Vaccination data

**Time to run**: ~3-4 minutes  
**Difficulty**: Intermediate

---

## 📦 Part 1: Setup and Imports

Let's import everything we need for this complete pipeline.

In [ ]:
import sys
sys.path.append('.')

# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Our utilities
from utils.load_data_utils import DataLoader
from utils.preprocess_data_utils import (
    aggregate_to_national,
    extract_national_mobility,
    merge_all_data
)
from utils.gluonts_utils import (
from utils.evaluation_utils import calculate_metrics, print_metrics
    create_gluonts_dataset,
    verify_dataset,
    prepare_train_test_split,
    get_feature_columns
)

# GluonTS
from gluonts.torch.model.deepar import DeepAREstimator
from gluonts.evaluation import make_evaluation_predictions

print("✓ All imports successful!")
print("  Ready to build COVID-19 forecasting pipeline")

---

## 📥 Part 2: Load COVID-19 Data

We'll load four data sources:
1. **Cases**: Daily confirmed cases by state
2. **Deaths**: Daily deaths by state  
3. **Mobility**: Google mobility data (6 metrics)
4. **Vaccines**: CDC vaccination data

All data is aggregated to the national level for this example.

In [ ]:
print("📥 Loading COVID-19 data...")
print("=" * 60)

# Initialize data loader
loader = DataLoader(data_dir="data")

# Load all data sources
cases_df = loader.load_cases()
deaths_df = loader.load_deaths()
mobility_df = loader.load_mobility()
vaccine_df = loader.load_vaccines()

print("\n✓ Data loaded successfully!")
print(f"  Cases shape:    {cases_df.shape}")
print(f"  Deaths shape:   {deaths_df.shape}")
print(f"  Mobility shape: {mobility_df.shape}")
print(f"  Vaccine shape:  {vaccine_df.shape}")
print("=" * 60)

---

## 🔧 Part 3: Preprocess and Aggregate

Now we'll:
1. Aggregate state-level data to national level
2. Calculate daily metrics and 7-day moving averages
3. Extract mobility metrics
4. Merge all data sources

**Why 7-day moving average?**
- Smooths out weekly reporting patterns
- Reduces noise (weekends have lower reporting)
- Better for forecasting

In [ ]:
print("🔧 Preprocessing data...")
print("=" * 60)

# 1. Aggregate cases to national level
print("\n1. Aggregating cases to national level...")
national_cases = aggregate_to_national(cases_df, data_type='cases')
print(f"   ✓ Cases aggregated: {len(national_cases)} days")

# 2. Aggregate deaths to national level
print("\n2. Aggregating deaths to national level...")
national_deaths = aggregate_to_national(deaths_df, data_type='deaths')
print(f"   ✓ Deaths aggregated: {len(national_deaths)} days")

# 3. Extract national mobility
print("\n3. Extracting national mobility data...")
national_mobility = extract_national_mobility(mobility_df)
print(f"   ✓ Mobility extracted: {len(national_mobility)} days")

# 4. Merge all data sources
print("\n4. Merging all data sources...")
merged_df = merge_all_data(
    national_cases,
    national_deaths,
    national_mobility,
    vaccine_df
)
print(f"   ✓ Data merged: {len(merged_df)} days")

print("\n" + "=" * 60)
print("✓ Preprocessing complete!")
print(f"\nFinal dataset:")
print(f"  Date range: {merged_df['Date'].min().date()} to {merged_df['Date'].max().date()}")
print(f"  Total days: {len(merged_df)}")
print(f"  Features: {len(merged_df.columns)-1} (excluding Date)")

# Show first few rows
print("\nFirst few rows:")
merged_df.head()

---

## 📊 Part 4: Explore the Data

Let's visualize what we're working with.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Plot 1: Daily Cases
axes[0].plot(merged_df['Date'], merged_df['Daily_Cases'], 
             alpha=0.3, label='Daily Cases (raw)', color='blue')
axes[0].plot(merged_df['Date'], merged_df['Daily_Cases_MA7'], 
             linewidth=2, label='7-Day Average', color='darkblue')
axes[0].set_title('US COVID-19 Daily Cases', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Cases')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Daily Deaths  
axes[1].plot(merged_df['Date'], merged_df['Daily_Deaths'], 
             alpha=0.3, label='Daily Deaths (raw)', color='red')
axes[1].plot(merged_df['Date'], merged_df['Daily_Deaths_MA7'], 
             linewidth=2, label='7-Day Average', color='darkred')
axes[1].set_title('US COVID-19 Daily Deaths', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Deaths')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Mobility (retail & recreation)
axes[2].plot(merged_df['Date'], merged_df['retail_and_recreation_percent_change_from_baseline'], 
             linewidth=2, label='Retail & Recreation', color='green')
axes[2].set_title('Google Mobility: Retail & Recreation', fontsize=14, fontweight='bold')
axes[2].set_ylabel('% Change')
axes[2].set_xlabel('Date')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].axhline(y=0, color='black', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('covid_data_overview.png', dpi=150, bbox_inches='tight')
print("✓ Plot saved as 'covid_data_overview.png'")
plt.show()

print("\n📊 Key Observations:")
print("  • Multiple waves of infections visible")
print("  • Deaths lag cases by ~2-3 weeks")
print("  • Mobility shows lockdown impacts")

---

## 🔧 Part 5: Prepare Data for GluonTS

Now we'll:
1. Split into train/test sets (80/20 split, last 14 days for testing)
2. Select features to use as exogenous variables
3. Convert to GluonTS format

**Target variable**: `Daily_Cases_MA7` (7-day moving average of cases)

**Exogenous features** (help improve predictions):
- Deaths metrics (daily, 7-day MA, CFR)
- Mobility metrics (6 mobility indicators)
- Cumulative cases/deaths

In [ ]:
print("🔧 Preparing data for GluonTS...")
print("=" * 60)

# Define target
TARGET = 'Daily_Cases_MA7'

# Define features to use (excluding target and date)
EXCLUDE_COLS = ['Date', TARGET, 'Daily_Cases', 'Cumulative_Cases', 'Daily_Deaths', 'Cumulative_Deaths']
past_features = get_feature_columns(merged_df, exclude_cols=EXCLUDE_COLS)

print(f"\nTarget variable: {TARGET}")
print(f"\nExogenous features ({len(past_features)}):")
for i, feat in enumerate(past_features, 1):
    print(f"  {i}. {feat}")

# Split train/test
print("\n" + "-" * 60)
train_df, test_df = prepare_train_test_split(
    merged_df,
    test_size=14,
    target_column=TARGET
)

print("\n" + "=" * 60)
print("✓ Data ready for GluonTS!")

### Convert to GluonTS Format

Now convert our pandas DataFrames to GluonTS `ListDataset` format.

In [ ]:
print("\n🔄 Converting to GluonTS format...")

# Create train dataset
train_ds = create_gluonts_dataset(
    df=train_df,
    target_column=TARGET,
    freq='D',
    prediction_length=14,
    past_feat_columns=past_features
)

# Create test dataset  
test_ds = create_gluonts_dataset(
    df=test_df,
    target_column=TARGET,
    freq='D',
    prediction_length=14,
    past_feat_columns=past_features
)

# Verify datasets
verify_dataset(train_ds, "Train")
verify_dataset(test_ds, "Test")

print("\n✓ GluonTS datasets ready!")

---

## 🤖 Part 6: Train DeepAR Model

Now for the main event - training DeepAR!

**Why DeepAR for COVID?**
- Handles **long-term dependencies** (waves span months)
- Works with **multiple features** (mobility, deaths, etc.)
- Produces **probabilistic forecasts** (uncertainty is crucial!)
- Captures **complex patterns** (seasonality, trends)

**Model Configuration**:
- `prediction_length=14`: Forecast 2 weeks ahead
- `context_length=60`: Use 2 months of history
- `num_layers=2`: Two RNN layers for pattern learning
- `hidden_size=40`: Network size (balanced)
- `epochs=20`: Training iterations (CPU-optimized)

**Training time**: ~2-3 minutes on CPU

In [ ]:
print("🏋️ Training DeepAR on COVID-19 data...")
print("=" * 60)

# Create estimator
estimator = DeepAREstimator(
    freq='D',
    prediction_length=14,       # Forecast 14 days
    context_length=60,          # Use 60 days of history
    num_layers=2,               # 2 RNN layers
    hidden_size=40,             # Hidden layer size
    dropout_rate=0.1,           # Prevent overfitting
    lr=0.001,                   # Learning rate
    batch_size=32,
    num_feat_dynamic_real=len(past_features),  # Number of features
    trainer_kwargs={
        'max_epochs': 20,       # CPU-optimized
        'enable_progress_bar': True,
        'enable_model_summary': False
    }
)

print("\n📚 Training started...")
print(f"  Using {len(past_features)} exogenous features")
print(f"  Training on {len(train_df)} days of data")
print(f"  This may take 2-3 minutes...\n")

predictor = estimator.train(train_ds)

print("\n" + "=" * 60)
print("✓ Training complete!")
print("  Model ready for forecasting!")

---

## 🔮 Part 7: Generate Forecasts

Let's use our trained model to forecast the next 14 days of COVID-19 cases!

**What we get**:
- Point forecast (mean prediction)
- Confidence intervals (10%, 25%, 50%, 75%, 90%)
- 100 sample paths (different possible futures)

In [ ]:
print("🔮 Generating forecasts...")
print("=" * 60)

# Generate predictions
forecast_it, ts_it = make_evaluation_predictions(
    dataset=test_ds,
    predictor=predictor,
    num_samples=100  # Generate 100 scenarios
)

forecasts = list(forecast_it)
ground_truths = list(ts_it)

forecast = forecasts[0]
actual = ground_truths[0]

print("✓ Forecasts generated!")
print(f"\n14-Day Forecast Summary:")
print(f"  Mean daily cases: {forecast.mean.mean():.0f}")
print(f"  Median: {np.median(forecast.mean):.0f}")
print(f"  Range: {forecast.mean.min():.0f} to {forecast.mean.max():.0f}")

print(f"\nPredictions by day:")
for i in range(14):
    mean_val = forecast.mean[i]
    low = forecast.quantile(0.1)[i]
    high = forecast.quantile(0.9)[i]
    print(f"  Day {i+1:2d}: {mean_val:7,.0f} cases  (80% CI: {low:7,.0f} - {high:7,.0f})")

print("\n" + "=" * 60)

---

## 📊 Part 8: Evaluate Performance

How well did our model do? Let's calculate comprehensive metrics.

In [ ]:
# Extract forecast period
forecast_period = 14
actual_values = actual[-forecast_period:]
forecast_values = forecast.mean

# Calculate metrics
def calculate_comprehensive_metrics(forecast, actual):
    """Calculate multiple forecasting metrics"""
    errors = forecast - actual
    
    # Basic metrics
    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors**2))
    mape = np.mean(np.abs(errors / actual)) * 100
    
    # Additional metrics
    me = np.mean(errors)  # Mean Error (bias)
    max_error = np.max(np.abs(errors))
    
    return {
        'mae': mae,
        'rmse': rmse, 
        'mape': mape,
        'me': me,
        'max_error': max_error
    }

metrics = calculate_comprehensive_metrics(forecast_values, actual_values)

print("\n📊 DeepAR Performance on COVID-19 Data:")
print("=" * 60)
print(f"MAE (Mean Absolute Error):      {metrics['mae']:>10,.0f} cases")
print(f"RMSE (Root Mean Squared Error): {metrics['rmse']:>10,.0f} cases")
print(f"MAPE (Mean Abs. % Error):       {metrics['mape']:>10.2f} %")
print(f"ME (Mean Error / Bias):         {metrics['me']:>10,.0f} cases")
print(f"Maximum Error:                   {metrics['max_error']:>10,.0f} cases")
print("=" * 60)

# Interpretation
if metrics['mape'] < 10:
    print("\n✓ Excellent! Error < 10%")
elif metrics['mape'] < 20:
    print("\n✓ Good performance! Error < 20%")
else:
    print("\n⚠️  Moderate performance (COVID is highly variable)")

if abs(metrics['me']) < metrics['mae'] / 2:
    print("✓ Low bias (model not systematically over/under-predicting)")
else:
    bias_direction = "over" if metrics['me'] > 0 else "under"
    print(f"⚠️  Model tends to {bias_direction}-predict")

---

## 📈 Part 9: Visualize Results

The moment of truth - let's see how our forecasts compare to reality!

In [ ]:
plt.figure(figsize=(16, 8))

# Historical training data (last 60 days for context)
train_context = train_df.tail(60)
plt.plot(train_context['Date'], train_context[TARGET],
         label='Historical Data (Last 60 days)', color='steelblue', 
         linewidth=2, alpha=0.8)

# Forecast dates
last_train_date = train_df['Date'].iloc[-1]
forecast_dates = pd.date_range(
    start=last_train_date + pd.Timedelta(days=1),
    periods=forecast_period,
    freq='D'
)

# Actual future values
plt.plot(forecast_dates, actual_values,
         label='Actual Cases', color='orange', linewidth=3, 
         marker='o', markersize=8, zorder=5)

# DeepAR forecast
plt.plot(forecast_dates, forecast_values,
         label='DeepAR Forecast', color='red', linewidth=3,
         marker='s', markersize=7, linestyle='--', zorder=4)

# Confidence intervals
plt.fill_between(
    forecast_dates,
    forecast.quantile(0.05),
    forecast.quantile(0.95),
    alpha=0.15, color='red', label='90% Confidence'
)

plt.fill_between(
    forecast_dates,
    forecast.quantile(0.25),
    forecast.quantile(0.75),
    alpha=0.25, color='red', label='50% Confidence'
)

# Formatting
plt.title('COVID-19 Case Forecasting with DeepAR (14-Day Horizon)', 
          fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=13)
plt.ylabel('Daily Cases (7-Day Moving Average)', fontsize=13)
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig('deepar_covid_forecast.png', dpi=150, bbox_inches='tight')
print("✓ Forecast plot saved as 'deepar_covid_forecast.png'")

plt.show()

---

## 🔍 Part 10: Forecast Analysis

Let's analyze the forecast quality and patterns.

In [ ]:
# Create detailed analysis plot
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Forecast vs Actual
axes[0, 0].plot(range(1, 15), actual_values, 'o-', 
                label='Actual', color='orange', linewidth=2, markersize=8)
axes[0, 0].plot(range(1, 15), forecast_values, 's--',
                label='Forecast', color='red', linewidth=2, markersize=7)
axes[0, 0].fill_between(range(1, 15), 
                         forecast.quantile(0.1), 
                         forecast.quantile(0.9),
                         alpha=0.2, color='red')
axes[0, 0].set_title('Forecast vs Actual (14 Days)', fontweight='bold')
axes[0, 0].set_xlabel('Day')
axes[0, 0].set_ylabel('Daily Cases (7-Day MA)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Forecast Errors
errors = forecast_values - actual_values
axes[0, 1].bar(range(1, 15), errors, color=['red' if e > 0 else 'green' for e in errors])
axes[0, 1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[0, 1].set_title('Daily Forecast Errors', fontweight='bold')
axes[0, 1].set_xlabel('Day')
axes[0, 1].set_ylabel('Error (Forecast - Actual)')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Plot 3: Absolute Percentage Errors
ape = np.abs(errors / actual_values) * 100
axes[1, 0].bar(range(1, 15), ape, color='steelblue', alpha=0.7)
axes[1, 0].axhline(y=metrics['mape'], color='red', linestyle='--', 
                    linewidth=2, label=f'Mean APE: {metrics["mape"]:.1f}%')
axes[1, 0].set_title('Absolute Percentage Error by Day', fontweight='bold')
axes[1, 0].set_xlabel('Day')
axes[1, 0].set_ylabel('Absolute % Error')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Plot 4: Uncertainty Width
ci_width = forecast.quantile(0.9) - forecast.quantile(0.1)
axes[1, 1].plot(range(1, 15), ci_width, 'o-', color='purple', linewidth=2, markersize=8)
axes[1, 1].set_title('Forecast Uncertainty (80% CI Width)', fontweight='bold')
axes[1, 1].set_xlabel('Day')
axes[1, 1].set_ylabel('CI Width (Cases)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('deepar_analysis.png', dpi=150, bbox_inches='tight')
print("✓ Analysis plot saved as 'deepar_analysis.png'")
plt.show()

print("\n📈 Analysis Insights:")
print(f"  • Average absolute error: {metrics['mae']:,.0f} cases")
print(f"  • Forecast tends to {'over' if metrics['me'] > 0 else 'under'}-predict by {abs(metrics['me']):,.0f} cases")
print(f"  • Uncertainty range: {ci_width.mean():,.0f} cases (average 80% CI width)")
print(f"  • Day with highest error: Day {np.argmax(np.abs(errors))+1} ({np.max(np.abs(errors)):,.0f} cases)")

---

## 🎓 Summary: Complete COVID-19 Forecasting Pipeline

Congratulations! You've built a complete forecasting system! 🎉

**What we accomplished**:
1. ✅ Loaded real COVID-19 data from multiple sources
2. ✅ Preprocessed and aggregated to national level
3. ✅ Engineered features (deaths, mobility, etc.)
4. ✅ Trained DeepAR with exogenous features
5. ✅ Generated 14-day probabilistic forecasts
6. ✅ Comprehensive evaluation with multiple metrics
7. ✅ Detailed visualization and analysis

**Key Findings**:

📊 **Data**:
- Multiple COVID waves with complex patterns
- Deaths lag cases by 2-3 weeks
- Mobility correlates with case trends

🤖 **Model**:
- DeepAR handles complex temporal patterns
- Exogenous features improve predictions
- Probabilistic forecasts quantify uncertainty

📈 **Performance**:
- See metrics above for specific numbers
- Confidence intervals capture uncertainty
- Model learns wave patterns

---

## 🚀 Next Steps & Improvements

### 1. Model Enhancements
```python
# Try different architectures
- Increase hidden_size (40 → 60) for more capacity
- Add more layers (2 → 3) for deeper patterns
- Tune context_length (60 → 90 days)
- Adjust learning rate and epochs
```

### 2. Feature Engineering
- Add more mobility metrics
- Include testing rates
- Add demographic factors
- Create interaction features

### 3. Scenario Analysis (Bonus)
- Simulate lockdown impacts
- Model vaccination effects
- Test intervention strategies

### 4. Model Comparison
- Run SimpleFeedForward (baseline)
- Run DeepNPTS (alternative)
- Compare metrics across models

### 5. Production Deployment
- Set up automated retraining
- Create monitoring dashboard
- Implement A/B testing
- Add alerting for anomalies

---

## 📚 Further Reading

**GluonTS Documentation**:
- [DeepAR Paper](https://arxiv.org/abs/1704.04110)
- [GluonTS Tutorials](https://ts.gluon.ai/stable/tutorials/index.html)

**COVID-19 Forecasting**:
- CDC Forecasting Hub
- COVID-19 Forecast Hub
- Epidemic forecasting best practices

**Time Series Resources**:
- Rob Hyndman's "Forecasting: Principles and Practice"
- Google's Time Series Forecasting Guide

---

**🎯 You now have a production-ready COVID-19 forecasting pipeline!**

Compare with SimpleFeedForward and DeepNPTS to see which performs best!